# InsuML benchmark

Runs InsuML with its default settings on the datasets in `datasets/`, several seeds per dataset. The metrics are those of a held-out test set (20%). Check the *Parameters* cell and *Run All*. Requires `requirements.txt`.

## Parameters

In [ ]:
import os

# True: one dataset (TEST_DATASET), its first seed, 300 s. False: all datasets and seeds, 1000 s.
TEST_MODE = True
TEST_DATASET = "diabetes"

TIME_LIMIT_S = 300 if TEST_MODE else 1000  # seconds per run

# Trials run in parallel; None: the default for this computer (set a number if RAM is short)
PARALLEL_TRIALS = None

# name: (folder inside datasets/, target column, seeds); one run per seed
DATASETS = {
    "blood-transfusion-service-center": ("Binary", "Class", [17, 30, 56, 476, 777]),
    "breast-w": ("Binary", "Class", [17, 30, 56, 131]),
    "credit-g": ("Binary", "class", [17, 19, 30, 56, 131, 300926]),
    "diabetes": ("Binary", "class", [4, 23, 56, 131, 300926]),
    "qsar-biodeg": ("Binary", "Class", [4, 21, 131, 4444, 300926]),
    "analcatdata_dmft": ("Classification", "Prevention", [4, 17, 21, 300926]),
    "cmc": ("Classification", "Contraceptive_method_used", [4, 21, 468]),
    "hypothyroid": ("Classification", "Class", [67, 468]),
    "mfeat-morphological": ("Classification", "class", [17, 67, 468, 4446]),
    "vehicle": ("Classification", "Class", [17, 42, 67, 112, 4446]),
    "cholesterol": ("Regression", "chol", [17, 42, 50, 4442, 4446]),
    "cloud": ("Regression", "TE", [42, 52, 78]),
    "liver-disorders": ("Regression", "drinks", [42, 87, 100, 476]),
    "plasma_retinol": ("Regression", "RETPLASMA", [30, 100, 476, 10343]),
}

# RESUME: skip finished datasets and restart partial ones. DATASET_RETRIES: extra attempts after a failed run.
RESUME = True
DATASET_RETRIES = 1

REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
DATASETS_DIR = os.path.join(REPO_ROOT, "datasets")
RESULTS_ROOT = os.path.join(os.getcwd(), "results")

In [ ]:
import sys

# common_utils.py lives in the parent folder
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "..")))
import json
import time

from common_utils import (
    disable_unsupported_gpu,
    cleanup_ludwig_artifacts,
    log_available_memory,
    LUDWIG_TMP_ROOT,
    load_dataset,
    save_parameters,
    save_environment_info,
    load_completed_runs,
    start_fresh_if_requested,
    archive_dataset_results,
    safe_metric_value,
    save_incremental_results,
)

# Must run before importing ludwig/torch (falls back to CPU on old GPUs)
disable_unsupported_gpu()

# Qt runs without a screen; logs and settings go to a scratch folder
os.environ.setdefault("QT_QPA_PLATFORM", "offscreen")
SCRATCH = os.path.join(os.path.dirname(LUDWIG_TMP_ROOT), ".insuml-tmp")  # outside .ludwig-tmp, which is purged around every run
os.environ["HAMELIN_WORKSPACE"] = os.path.join(SCRATCH, "workspace")
sys.path.insert(0, os.path.join(REPO_ROOT, "src"))

from PySide6.QtWidgets import QApplication
qt_app = QApplication.instance() or QApplication([])
from hamelin.model.data_model import DataModel
from hamelin.view.pages.training_page import TrainingPage
from hamelin.analytics.ludwig_trainer import _run_backend

# Ludwig's trial dumps; emptied around every run
OUTPUT_DIR = os.path.join(SCRATCH, "output")
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Defaults of a fresh training page
_page = TrainingPage()
APP_TIME_BUDGET_S = int(_page.time_budget.value())
APP_TEST_SPLIT = float(_page.test_split.value())
APP_SEED = int(_page.random_seed.value())
APP_PARALLEL_TRIALS = int(_page.parallel_trials.value())
_page.deleteLater()
del _page

if TEST_DATASET not in DATASETS:
    raise ValueError(f"TEST_DATASET '{TEST_DATASET}' is not in DATASETS.")

# results/<RUN_MODE_NAME>/
RUN_MODE_NAME = f"{'test' if TEST_MODE else 'full'}_{TIME_LIMIT_S}s_test{round(APP_TEST_SPLIT * 100)}pct"
DATASET_NAMES = [TEST_DATASET] if TEST_MODE else list(DATASETS)
# Seeds each dataset is run with (the first one only in test mode)
SEEDS_TO_RUN = {name: DATASETS[name][2][:1] if TEST_MODE else DATASETS[name][2] for name in DATASET_NAMES}

# Saved as parameters.json
PARAMETERS = {
    "run_mode": RUN_MODE_NAME,
    "test_mode": TEST_MODE,
    "tool": "InsuML",
    "time_limit_s": TIME_LIMIT_S,
    "app_default_time_limit_s": APP_TIME_BUDGET_S,
    "test_fraction": APP_TEST_SPLIT,
    "app_default_seed": APP_SEED,
    "parallel_trials": PARALLEL_TRIALS or APP_PARALLEL_TRIALS,
    "parallel_trials_overridden": PARALLEL_TRIALS is not None,
    "resume": RESUME,
    "dataset_retries": DATASET_RETRIES,
    "datasets_dir": DATASETS_DIR,
    "datasets": {name: list(DATASETS[name][:2]) for name in DATASET_NAMES},
    "seeds_per_dataset": SEEDS_TO_RUN,
}
print(f"Run mode '{RUN_MODE_NAME}': {len(DATASET_NAMES)} dataset(s), "
      f"{sum(len(s) for s in SEEDS_TO_RUN.values())} run(s) of up to {TIME_LIMIT_S} s each.")

## InsuML experiment functions

In [ ]:
EXPECTED_PROBLEM_TYPE = {"Binary": "binary", "Classification": "category", "Regression": "number"}


def open_training_page(name, target_column, group):
    """Opens the training page on a dataset with the outcome and all predictors selected."""
    data = load_dataset(name, DATASETS, DATASETS_DIR)  # existence and target column check
    if data is None:
        return None, None
    dm = DataModel()
    dm.load_from_file(os.path.join(DATASETS_DIR, group, f"{name}.csv"))  # type detection, missing markers
    page = TrainingPage()
    page.set_data_model(dm)
    page.primary_combo.setCurrentText(target_column)
    if page.primary_combo.currentText() != target_column:
        raise ValueError(f"'{target_column}' is not a column of {name}.")
    page._select_all_predictors()
    suggested = page.problem_type_combo.currentIndex()
    from hamelin.view.pages.training_page import _PROBLEM_TYPES
    problem_type = _PROBLEM_TYPES[suggested][1]
    if problem_type != EXPECTED_PROBLEM_TYPE[group]:
        raise ValueError(f"The application suggests '{problem_type}' for {target_column}, "
                         f"not '{EXPECTED_PROBLEM_TYPE[group]}' ({group}).")
    return page, dm


def page_run_arguments(page):
    """Outcome, predictors, time budget and backend arguments read from the page."""
    target = page.primary_combo.currentText().strip()
    features = [item.text() for item in page.features_list.selectedItems() if item.text() != target]
    backend_kwargs = {
        "test_split": float(page.test_split.value()),
        "random_seed": int(page.random_seed.value()),
        "output_directory": OUTPUT_DIR,
    }
    fields = page._collect_config_fields()
    if fields:
        backend_kwargs["user_config_fields"] = fields
    return target, features, int(page.time_budget.value()), backend_kwargs


def train_insuml(page, seed):
    """Starts one training with this run's seed."""
    if PARALLEL_TRIALS is not None:
        page.parallel_trials.setValue(PARALLEL_TRIALS)
    page.time_budget.setValue(TIME_LIMIT_S)
    page.random_seed.setValue(seed)
    target, features, time_limit_s, backend_kwargs = page_run_arguments(page)
    return _run_backend(
        df=page._rows_after_rules(),
        target=target,
        features=features,
        time_limit_s=time_limit_s,
        backend=None,  # default backend
        backend_kwargs=backend_kwargs,
    )


def extract_diagnostics(result, fields):
    """Model, trials, normalisation and split of one run, for auditing."""
    diag = {"model_type": result.model_type, "n_trials": result.num_trials,
            "training_time_s": result.training_time_seconds,
            "page_fields": fields,
            "best_hyperparameters": result.hyperparameters}
    try:
        cfg = result.trained_model.config
        trainer = cfg.get("trainer") or {}
        diag["combiner"] = (cfg.get("combiner") or {}).get("type")
        diag["trainer"] = {k: trainer.get(k) for k in (
            "epochs", "batch_size", "learning_rate", "optimizer", "early_stop")}
        diag["output_features"] = {
            f["name"]: {"type": f["type"], "normalization": (f.get("preprocessing") or {}).get("normalization")}
            for f in cfg["output_features"]}
        diag["split"] = (cfg.get("preprocessing") or {}).get("split")
    except Exception as e:
        print(f"[WARN] Could not read the model's settings: {e}")
    return diag


def extract_metrics(test_metrics):
    """Numeric test metrics of the outcome."""
    metrics = {m: safe_metric_value(v) for m, v in test_metrics.items()}
    return {m: float(v) for m, v in metrics.items() if isinstance(v, (int, float))}


def check_output_type(result, target_column, task_type):
    """Fails if the outcome type does not match the task."""
    inferred = next(f["type"] for f in result.trained_model.config["output_features"] if f["name"] == target_column)
    expected = ("category", "binary") if task_type == "classification" else ("number",)
    if inferred not in expected:
        raise ValueError(f"The model's outcome is a '{inferred}' for a {task_type} dataset.")


def run_insuml_experiment(page, target_column, seed, task_type):
    """One training: (metrics, diagnostics)."""
    cleanup_ludwig_artifacts(OUTPUT_DIR)  # stale hyperopt/ from a previous run
    log_available_memory()
    try:
        result = train_insuml(page, seed)
        check_output_type(result, target_column, task_type)
        metrics = extract_metrics(result.test_metrics)
        if not metrics:
            raise RuntimeError("The training returned no test metrics (its evaluation failed).")
        metrics["train_time_s"] = round(float(result.training_time_seconds), 2)  # config + search + training
        metrics["n_trials"] = int(result.num_trials)
        diagnostics = extract_diagnostics(result, page_run_arguments(page)[3].get("user_config_fields"))
        del result
        return metrics, diagnostics
    except Exception as e:
        print(f"[ERROR] InsuML, seed {seed}: {e}")
        return None, None
    finally:
        # Metrics are already in memory; trial dumps are no longer needed
        cleanup_ludwig_artifacts(OUTPUT_DIR)

## Main loop

In [ ]:
# results/<run mode>/InsuML/
save_dir = os.path.join(RESULTS_ROOT, RUN_MODE_NAME, "InsuML")
start_fresh_if_requested(save_dir, RESUME)  # only acts with RESUME = False
os.makedirs(save_dir, exist_ok=True)
save_parameters(save_dir, PARAMETERS)
save_environment_info(save_dir, PARAMETERS, RESUME, tool="InsuML")
diagnostics_log_path = os.path.join(save_dir, "InsuML_run_diagnostics.json")


def load_json_log(path):
    """Previous log, to keep what already-finished runs recorded when resuming."""
    if RESUME and os.path.isfile(path):
        try:
            with open(path) as f:
                return json.load(f)
        except Exception as e:
            print(f"[WARN] Could not read {os.path.basename(path)}: {e}")
    return {}


def save_json_log(path, log):
    try:
        with open(path, "w") as f:
            json.dump(log, f, indent=2, default=str)
    except Exception as e:
        print(f"[WARN] Could not save {os.path.basename(path)}: {e}")


def update_analysis():
    """Refreshes analysis/ with the datasets done so far, so the summary grows while the benchmark runs."""
    try:
        from analyze_results import run_analysis  # lives in the parent folder
        run_analysis(os.path.join(RESULTS_ROOT, RUN_MODE_NAME), verbose=False, tool="InsuML")
    except (Exception, SystemExit) as e:
        print(f"[WARN] Could not update the analysis ({e})")


diagnostics_log = load_json_log(diagnostics_log_path)

for name in DATASET_NAMES:
    print(f"\n=== Dataset {name} ===")
    data = load_dataset(name, DATASETS, DATASETS_DIR)  # task type and group of the dataset
    if data is None:
        continue
    page, dm = open_training_page(name, data["target_column"], data["group"])
    if page is None:
        continue
    seeds = SEEDS_TO_RUN[name]  # one run per seed
    print(f"[INFO] {name}: target '{data['target_column']}', {dm.n_rows} rows, {len(seeds)} run(s) planned.")

    def forget_dataset_logs():
        """Drops what the discarded runs of this dataset recorded in the JSON logs."""
        for key in [k for k in diagnostics_log if k.startswith(f"{name}_seed")]:
            del diagnostics_log[key]
        save_json_log(diagnostics_log_path, diagnostics_log)

    # A finished dataset is skipped; a half-done one is discarded and restarted from its first seed
    runs = load_completed_runs(save_dir, name, len(seeds), RESUME)
    if len(runs) < len(seeds):
        forget_dataset_logs()

    for attempt in range(1 + DATASET_RETRIES):
        if len(runs) == len(seeds):
            break
        if attempt > 0:
            print(f"[RETRY] {name}: a run failed; discarding this dataset's runs "
                  f"and starting it over (attempt {attempt + 1} of {1 + DATASET_RETRIES})")
            archive_dataset_results(save_dir, name)
            forget_dataset_logs()
            runs = []
        for seed in seeds:
            print(f"[RUN] {name} seed={seed}")
            run_start = time.perf_counter()
            metrics, diagnostics = run_insuml_experiment(page, data["target_column"], seed, data["task_type"])
            run_time = time.perf_counter() - run_start
            if not metrics:
                break  # a failed run: the whole dataset is retried
            metrics["time_taken"] = round(run_time, 2)  # seconds, train + eval
            runs.append({"seed": seed, "metrics": metrics})
            run_key = f"{name}_seed{seed}"
            if diagnostics:
                diagnostics_log[run_key] = diagnostics
            # Saved after every run so an interruption loses nothing
            save_json_log(diagnostics_log_path, diagnostics_log)
            save_incremental_results(name, data["group"], runs, save_dir)
    if len(runs) < len(seeds):
        print(f"[WARN] {name}: only {len(runs)} of {len(seeds)} runs completed after "
              f"{1 + DATASET_RETRIES} attempt(s). Launch the benchmark again to restart it.")
    update_analysis()  # summary_metrics.csv now includes this dataset


print("\nAll datasets processed.")
print(f"Results in: {save_dir}")
print(f"Per-run diagnostics (model, trials, outcome normalisation): {diagnostics_log_path}")


## Analysis

In [ ]:
# Which analysis files to write; the "mean ± sd" table is always shown below.
# Available: summary_metrics, datasets_summary.
ANALYSIS_OUTPUTS = ["summary_metrics", "datasets_summary"]

try:
    from IPython.display import display
    from analyze_results import run_analysis  # lives in the parent folder

    summary = run_analysis(os.path.join(RESULTS_ROOT, RUN_MODE_NAME), outputs=ANALYSIS_OUTPUTS, tool="InsuML")
    display(summary)
except (Exception, SystemExit) as e:
    print(f"[WARN] The analysis could not run ({e}). From a terminal: "
          f"python analyze_results.py {RUN_MODE_NAME} --tool InsuML")
